# Architecture Ablation

Runs the architecture sweep with optional reaction-force loss and optional Hessian regularization. Edit the config cell before running.

In [1]:
import os
import numpy as np

from properties import SlinkyN3Properties
from run_architectures import (
    SweepConfig,
    run_architecture_sweep,
    subset_energy_only,
    subset_main_paper_candidates,
    subset_all,
    plot_summary_final_losses,
)


In [2]:
# Dataset / physical setup
properties = SlinkyN3Properties(mass=0.3)
train_file = "../simulation_data_2D/3_noded/n3_slinky_sim_train_dataset.npz"
valid_file = "../simulation_data_2D/3_noded/n3_slinky_sim_test_dataset.npz"

# Pick one subset, or pass None to run all registered architectures.
# selected_architectures = subset_energy_only()
# selected_architectures = subset_main_paper_candidates()
selected_architectures = subset_all()
# selected_architectures = None

# Optional force loss. Leave strength at 0.0 for displacement-only training.
force_loss_strength = 1
force_key = None          # None auto-detects F/forces/reaction_force when force loss is enabled
force_components = (0,)   # e.g. (0,) for Fx, or (0, 1, 2)
force_sign = 1.0
return_loss_components = force_loss_strength != 0.0

# Optional Hessian regularizer. Leave strength at 0.0 to disable.
hessian_reg_strength = 0.0
hessian_reg_probes = 1
hessian_reg_seed = 0


In [3]:
cfg = SweepConfig(
    output_dir="arch_ablation_outputs_n3_slinky_simdata",
    n_epochs=500,
    lr=1e-2,
    seed=42,
    hidden=(10,),
    input_mode="invariant",
    activation="tanh",
    corr_factor=0.01,
    only_stretching_NN=False,
    zero_reference=True,
    valid_every=1,
    max_dlambda=5e-2,
    iters=20,
    ls_steps=10,
    abs_tol=1e-4,
    rel_tol=1e-4,
    train_fail_on_nonconvergence=False,
    prediction_fail_on_nonconvergence=False,
    hessian_reg_strength=hessian_reg_strength,
    hessian_reg_probes=hessian_reg_probes,
    hessian_reg_seed=hessian_reg_seed,
    force_key=force_key,
    force_loss_strength=force_loss_strength,
    force_components=force_components,
    force_sign=force_sign,
    return_loss_components=return_loss_components,
    save_npz=True,
    save_model=True,
    save_plots=True,
    save_energy_landscapes=True,
    verbose=True,
)


In [4]:
results = run_architecture_sweep(
    properties=properties,
    train_file=train_file,
    valid_file=valid_file,
    cfg=cfg,
    selected_architectures=selected_architectures,
)


Running architecture: diag_energy_baseline
  model_cls               : DiagonalPlusEnergyNN
  which_case              : baseline
  hidden                  : (10,)
  input_mode              : invariant
  only_stretching_NN      : False
  only_bending_NN         : False
  activation              : tanh
  corr_factor             : 0.01
  zero_reference          : True
  seed                    : 42
  max_dlambda             : 0.05
  iters                   : 20
  ls_steps                : 10
  abs_tol                 : 0.0001
  rel_tol                 : 0.0001
  training fail_on_nonconvergence       : False
  validation loss fail_on_nonconvergence: False
  prediction fail_on_nonconvergence     : False
  hessian_reg_strength    : 0.0
  hessian_reg_probes      : 1
  hessian_reg_seed        : 0
  force_key               : None
  force_loss_strength     : 1
  force_components        : (0,)
  force_sign              : 1.0
  exp_dir                 : arch_ablation_outputs_n3_slinky_simdata/diag

/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/architecture_plots.py:234: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


Running architecture: diag_energy_mlp
  model_cls               : DiagonalPlusEnergyNN
  which_case              : MLP
  hidden                  : (10,)
  input_mode              : invariant
  only_stretching_NN      : False
  only_bending_NN         : False
  activation              : tanh
  corr_factor             : 0.01
  zero_reference          : True
  seed                    : 42
  max_dlambda             : 0.05
  iters                   : 20
  ls_steps                : 10
  abs_tol                 : 0.0001
  rel_tol                 : 0.0001
  training fail_on_nonconvergence       : False
  validation loss fail_on_nonconvergence: False
  prediction fail_on_nonconvergence     : False
  hessian_reg_strength    : 0.0
  hessian_reg_probes      : 1
  hessian_reg_seed        : 0
  force_key               : None
  force_loss_strength     : 1
  force_components        : (0,)
  force_sign              : 1.0
  exp_dir                 : arch_ablation_outputs_n3_slinky_simdata/diag_energy_ml

In [6]:
summary_plot = os.path.join(cfg.output_dir, "final_loss_summary.png")
plot_summary_final_losses(results, save_path=summary_plot, show=False)
summary_plot


NameError: name 'plt' is not defined

In [8]:
from architecture_plots import plot_architecture_comparison_paper
plot_dir = cfg.output_dir
# good architectures
# plot_architectures = selected_architectures
plot_architectures = \
[
    "diag_energy_baseline",
    "chol_energy_baseline",
    "diag_energy_icnn",
    "chol_energy_icnn",
    # add whatever you want here
]

plot_architecture_comparison_paper(
    architectures=plot_architectures,
    # architectures=subset_all(),
    results_dir=plot_dir,
    traj_idx=2,
)


/var/folders/z0/3frv2l990hb5ryd49z8vm4w80000gn/T/ipykernel_34819/671625886.py:14: UserWarning: traj_idx=2 is unavailable for split 'train' (available trajectories: 0..1). That figure will be left empty while the other split is still plotted if available.
  plot_architecture_comparison_paper(


{'training_loss': 'arch_ablation_outputs_n3_slinky_simdata/paper_ready_architecture_comparison/training_loss_comparison.pdf',
 'validation_loss': 'arch_ablation_outputs_n3_slinky_simdata/paper_ready_architecture_comparison/validation_loss_comparison.pdf',
 'training_trajectory': 'arch_ablation_outputs_n3_slinky_simdata/paper_ready_architecture_comparison/training_trajectory_comparison.pdf',
 'validation_trajectory': 'arch_ablation_outputs_n3_slinky_simdata/paper_ready_architecture_comparison/validation_trajectory_comparison.pdf',
 'training_xz_trajectory': 'arch_ablation_outputs_n3_slinky_simdata/paper_ready_architecture_comparison/training_xz_trajectory_comparison.pdf',
 'validation_xz_trajectory': 'arch_ablation_outputs_n3_slinky_simdata/paper_ready_architecture_comparison/validation_xz_trajectory_comparison.pdf',
 'combined_xz_trajectory': 'arch_ablation_outputs_n3_slinky_simdata/paper_ready_architecture_comparison/combined_xz_trajectory_comparison.pdf',
 'colors': {'diag_energy_bas